In [2]:
!git clone https://github.com/sscardapane/broken-repo.git

Cloning into 'broken-repo'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), done.


In [3]:
%cd broken-repo

/content/broken-repo


In [4]:
!python3 -m unittest discover -s tests

F..F.
FAIL: test_completed_task_disappears_from_default_list (test_cli.CliTest.test_completed_task_disappears_from_default_list)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/broken-repo/tests/test_cli.py", line 35, in test_completed_task_disappears_from_default_list
    self.assertIn("No tasks.", list_result.stdout)
AssertionError: 'No tasks.' not found in '1. [ ] Fix failing tests\n'

FAIL: test_complete_persists_done_status (test_store.TaskStoreTest.test_complete_persists_done_status)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/broken-repo/tests/test_store.py", line 32, in test_complete_persists_done_status
    self.assertTrue(reloaded.get(1).done)
AssertionError: False is not true

----------------------------------------------------------------------
Ran 5 tests in 0.390s

FAILED (failures=2)


## Minimal local LLM client

For Colab with a T4 GPU, run the next cells to load `Qwen/Qwen3.5-2B` inside the Colab VM. The helper in `utils.py` starts a local OpenAI-compatible server on `127.0.0.1`, so inference stays on the remote VM and does not use a hosted API.

For macOS, the simplest path is Ollama with a smaller coding model. In a terminal, run:

```bash
ollama pull qwen2.5-coder:1.5b
```

Then call `load_llm(model="qwen2.5-coder:1.5b", base_url="http://127.0.0.1:11434/v1", start_server=False)` instead.

In [ ]:
# Colab/T4 only: install the client and the lightweight Transformers server.
# This can take a few minutes because Qwen3.5 currently needs recent Transformers support.
!pip -q install -U openai pillow torchvision
!pip -q install -U "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

In [ ]:
# Make utils.py importable. If this notebook was opened directly from GitHub in
# Colab, sibling files are not always copied into the runtime automatically.
import sys
import urllib.request
from pathlib import Path

UTILS_URL = "https://raw.githubusercontent.com/sscardapane/coding-harness-demo/main/utils.py"

for candidate in (Path.cwd(), Path.cwd().parent, Path("/content")):
    if (candidate / "utils.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    utils_path = Path("/content/utils.py")
    urllib.request.urlretrieve(UTILS_URL, utils_path)
    sys.path.insert(0, str(utils_path.parent))

In [ ]:
from utils import load_llm

# This starts Qwen/Qwen3.5-2B on 127.0.0.1:8000 and returns a callable helper.
llm = load_llm()

print(llm("In one sentence, say what a failing unit test is."))